<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 11. BERT: MLM과 NSP — BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding

- **원 논문:** [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/abs/1810.04805)
- **저자 / 발표:** Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova · NAACL 2019 (2018)
- **난이도 / 예상 시간:** 중상급 · 약 90분
- **선수 지식:** Transformer encoder, masked loss, segment embedding, multi-task learning

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

token·position·segment embedding을 더한 양방향 Transformer encoder를 만들고, 선택 token의 80/10/10 corruption을 거친 Masked Language Model과 [CLS] 기반 Next Sentence Prediction을 공동 학습한다. 두 metric과 오른쪽 문맥 민감도를 검증한다.

**원 논문과 다른 것**

BERT Base/Large는 각각 12/24층, 110M/340M 파라미터로 BooksCorpus와 English Wikipedia(총 33억 단어)에서 100만 step 사전학습됐다. 여기서는 10개 문장 쌍, 1층 24-hidden encoder를 100여 step 암기시키는 구조 검증이다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3: input representation = token + segment + position embedding | `TinyBERT.embed`의 세 embedding 합과 LayerNorm | hidden 및 두 head logit shape 검사 |
| §3.1, Task #1: Masked LM에서 입력 token의 15%를 선택 | 특수 token을 제외한 60개 중 10개를 고정 선택(미니배치 근사 16.7%) | 선택 수와 특수 token 비선택을 assert |
| §3.1, MLM 선택 token의 80% [MASK], 10% random, 10% unchanged | `corrupted_ids`의 deterministic 8/1/1 corruption | 각 corruption 개수를 정확히 검사 |
| §3.1, Task #2: [CLS]로 IsNext/NotNext 예측 | `nsp_head(hidden[:, 0])`와 binary CE | NSP accuracy 출력 및 assert |
| Figure 1: GPT와 달리 모든 층에서 좌우 문맥을 함께 조건으로 사용 | mask 없는 `BidirectionalSelfAttention`과 오른쪽 token 교란 실험 | right-context sensitivity가 0보다 큰지 assert |
| §3 및 Figure 1: Transformer encoder의 양방향 self-attention | `BidirectionalSelfAttention`과 `BertEncoderBlock`의 post-LN residual | attention `[B,H,L,L]`과 양방향 확률 질량을 검사 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\mathcal{L}_{\mathrm{BERT}}=
\mathcal{L}_{\mathrm{MLM}}+\mathcal{L}_{\mathrm{NSP}},\qquad
\mathcal{L}_{\mathrm{MLM}}=-\sum_{i\in\mathcal{M}}
\log p(x_i\mid x_{\setminus\mathcal{M}})
$$

- $\mathcal{M}$은 masking된 token 집합이고 NSP는 두 segment의 연속 여부를 판별합니다.
- bidirectional attention이 mask 좌·우 context를 모두 사용하는지 단방향 기준과 비교합니다.
- token/position/segment embedding, encoder, MLM/NSP head를 별도 모듈로 나눕니다.
- encoder 출력 $[B,T,D]$에서 MLM $[B,T,V]$와 NSP $[B,2]$를 계산합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3: input representation = token + segment + position embedding
- **노트북 구현:** `TinyBERT.embed`의 세 embedding 합과 LayerNorm
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** hidden 및 두 head logit shape 검사를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** BERT Base/Large는 각각 12/24층, 110M/340M 파라미터로 BooksCorpus와 English Wikipedia(총 33억 단어)에서 100만 step 사전학습됐다. 여기서는 10개 문장 쌍, 1층 24-hidden encoder를 100여 step 암기시키는 구조 검증이다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 핵심 아이디어

BERT는 causal mask가 없는 Transformer encoder로 모든 층에서 왼쪽과 오른쪽 문맥을
함께 사용한다. 그대로 다음 token을 예측하면 정답을 보게 되므로, 입력 token 일부를
가리고 원 token을 맞히는 MLM을 쓴다. 선택된 token은 80% `[MASK]`, 10% random,
10% unchanged로 바꾸며, 문장 쌍의 `[CLS]` 표현에는 NSP 이진 목적을 함께 적용한다.

구현에서는 `nn.TransformerEncoder`를 한 줄로 호출하지 않는다. query/key/value
projection, head 분할, scaled score, 전체 위치 softmax를
`BidirectionalSelfAttention`으로 드러낸다. `BertEncoderBlock`은 BERT의 원래
post-LN 순서인 `LayerNorm(x + sublayer(x))`를 attention과 GELU FFN 뒤에 각각
적용한다. causal mask가 없다는 사실이 코드에서 바로 보여야 GPT와의 차이를
설명할 수 있다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** tokens [B,T] → hidden [B,T,D] → MLM [B,T,V], NSP [B,2]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

SEED = 1111
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
print(ACCELERATOR.summary())
PAD, CLS, SEP, MASK = 0, 1, 2, 3
VOCAB_SIZE = 24

sentence_pairs = [
    ([4, 5, 6], [7, 8, 9], 1),
    ([7, 8, 9], [13, 14, 15], 1),
    ([10, 11, 12], [16, 17, 18], 1),
    ([16, 17, 18], [19, 20, 21], 1),
    ([4, 6, 5], [8, 9, 7], 1),
    ([4, 5, 6], [16, 17, 18], 0),
    ([7, 8, 9], [19, 20, 21], 0),
    ([10, 11, 12], [7, 8, 9], 0),
    ([16, 17, 18], [13, 14, 15], 0),
    ([4, 6, 5], [19, 21, 20], 0),
]
input_ids = torch.tensor([[CLS, *a, SEP, *b, SEP] for a, b, _ in sentence_pairs])
segment_ids = torch.tensor([[0, 0, 0, 0, 0, 1, 1, 1, 1]] * len(sentence_pairs))
nsp_labels = torch.tensor([label for _, _, label in sentence_pairs])
input_ids, segment_ids, nsp_labels = ACCELERATOR.move(
    input_ids, segment_ids, nsp_labels
)

### 구현 단계 2 · 논문 주장 확인 실험

- **원문 위치:** §3.1, Task #1: Masked LM에서 입력 token의 15%를 선택 / §3.1, MLM 선택 token의 80% [MASK], 10% random, 10% unchanged
- **이 셀의 책임:** 특수 token을 제외한 60개 중 10개를 고정 선택(미니배치 근사 16.7%) / `corrupted_ids`의 deterministic 8/1/1 corruption
- **Tensor shape 계약:** tokens [B,T] → hidden [B,T,D] → MLM [B,T,V], NSP [B,2]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** 선택 수와 특수 token 비선택을 assert / 각 corruption 개수를 정확히 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
selected_positions = torch.tensor([1, 2, 3, 5, 6, 7, 1, 2, 5, 6], device=DEVICE)
selected_rows = torch.arange(len(input_ids), device=DEVICE)
corrupted_ids = input_ids.clone()
mlm_labels = torch.full_like(input_ids, -100)
mlm_labels[selected_rows, selected_positions] = input_ids[
    selected_rows, selected_positions
]

corrupted_ids[selected_rows[:8], selected_positions[:8]] = MASK
corrupted_ids[selected_rows[8], selected_positions[8]] = 22  # random token
# row 9 is deliberately unchanged

chosen = mlm_labels != -100
mask_count = int((corrupted_ids[chosen] == MASK).sum())
random_count = int(
    (
        (corrupted_ids[chosen] != MASK) & (corrupted_ids[chosen] != input_ids[chosen])
    ).sum()
)
unchanged_count = int((corrupted_ids[chosen] == input_ids[chosen]).sum())
assert int(chosen.sum()) == 10
assert (mask_count, random_count, unchanged_count) == (8, 1, 1)
assert not chosen[:, 0].any() and not chosen[:, 4].any() and not chosen[:, 8].any()
print(f"selected={int(chosen.sum())}/60 content tokens; corruption=8/1/1")

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3: input representation = token + segment + position embedding / Figure 1: GPT와 달리 모든 층에서 좌우 문맥을 함께 조건으로 사용 / §3 및 Figure 1: Transformer encoder의 양방향 self-attention
- **이 셀의 책임:** `TinyBERT.embed`의 세 embedding 합과 LayerNorm / mask 없는 `BidirectionalSelfAttention`과 오른쪽 token 교란 실험 / `BidirectionalSelfAttention`과 `BertEncoderBlock`의 post-LN residual
- **Tensor shape 계약:** tokens [B,T] → hidden [B,T,D] → MLM [B,T,V], NSP [B,2]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** hidden 및 두 head logit shape 검사 / right-context sensitivity가 0보다 큰지 assert / attention `[B,H,L,L]`과 양방향 확률 질량을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class BidirectionalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model % num_heads:
            raise ValueError("d_model must be divisible by num_heads")
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query = nn.Linear(d_model, d_model, bias=False)
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)
        self.output = nn.Linear(d_model, d_model, bias=False)

    def _split_heads(self, tensor):
        batch, length, _ = tensor.shape
        split = tensor.view(
            batch,
            length,
            self.num_heads,
            self.head_dim,
        )
        return split.transpose(1, 2)

    def _merge_heads(self, tensor):
        batch, _, length, _ = tensor.shape
        merged = tensor.transpose(1, 2).contiguous()
        return merged.view(
            batch,
            length,
            self.num_heads * self.head_dim,
        )

    def forward(self, hidden):
        query = self._split_heads(self.query(hidden))
        key = self._split_heads(self.key(hidden))
        value = self._split_heads(self.value(hidden))
        scores = query @ key.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)
        weights = torch.softmax(scores, dim=-1)
        context = self._merge_heads(weights @ value)
        return self.output(context), weights


class BertEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attention = BidirectionalSelfAttention(
            d_model,
            num_heads,
        )
        self.attention_norm = nn.LayerNorm(d_model)
        self.intermediate = nn.Linear(d_model, d_ff)
        self.output = nn.Linear(d_ff, d_model)
        self.output_norm = nn.LayerNorm(d_model)

    def forward(self, hidden):
        attended, weights = self.attention(hidden)
        hidden = self.attention_norm(hidden + attended)
        transformed = self.output(F.gelu(self.intermediate(hidden)))
        hidden = self.output_norm(hidden + transformed)
        return hidden, weights


class TinyBERT(nn.Module):
    def __init__(self, vocab_size, max_length, d_model=24, num_heads=4):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.segment_embedding = nn.Embedding(2, d_model)
        self.embedding_norm = nn.LayerNorm(d_model)
        self.encoder = BertEncoderBlock(
            d_model,
            num_heads,
            d_ff=2 * d_model,
        )
        self.mlm_transform = nn.Linear(d_model, d_model)
        self.mlm_norm = nn.LayerNorm(d_model)
        self.mlm_bias = nn.Parameter(torch.zeros(vocab_size))
        self.nsp_head = nn.Linear(d_model, 2)

    def embed(self, token_ids, token_type_ids):
        batch, length = token_ids.shape
        positions = torch.arange(
            length,
            device=token_ids.device,
        ).unsqueeze(0)
        positions = positions.expand(batch, length)
        return self.embedding_norm(
            self.token_embedding(token_ids)
            + self.position_embedding(positions)
            + self.segment_embedding(token_type_ids)
        )

    def masked_language_model_logits(self, hidden):
        transformed = F.gelu(self.mlm_transform(hidden))
        transformed = self.mlm_norm(transformed)
        return F.linear(
            transformed,
            self.token_embedding.weight,
            self.mlm_bias,
        )

    def forward(self, token_ids, token_type_ids):
        embedded = self.embed(token_ids, token_type_ids)
        hidden, weights = self.encoder(embedded)
        mlm_logits = self.masked_language_model_logits(hidden)
        nsp_logits = self.nsp_head(hidden[:, 0])
        return mlm_logits, nsp_logits, hidden, weights


torch.manual_seed(SEED)
model = TinyBERT(VOCAB_SIZE, max_length=input_ids.shape[1]).to(DEVICE)
probe_mlm, probe_nsp, probe_hidden, probe_attention = model(
    corrupted_ids,
    segment_ids,
)
assert probe_mlm.shape == (*input_ids.shape, VOCAB_SIZE)
assert probe_nsp.shape == (len(input_ids), 2)
assert probe_hidden.shape[:2] == input_ids.shape
assert probe_attention.shape == (len(input_ids), 4, 9, 9)
expected_attention_sums = probe_attention.new_ones(
    len(input_ids),
    4,
    9,
)
assert torch.allclose(
    probe_attention.sum(dim=-1),
    expected_attention_sums,
    atol=1e-6,
)

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §3.1, Task #1: Masked LM에서 입력 token의 15%를 선택 / §3.1, MLM 선택 token의 80% [MASK], 10% random, 10% unchanged / §3.1, Task #2: [CLS]로 IsNext/NotNext 예측
- **이 셀의 책임:** 특수 token을 제외한 60개 중 10개를 고정 선택(미니배치 근사 16.7%) / `corrupted_ids`의 deterministic 8/1/1 corruption / `nsp_head(hidden[:, 0])`와 binary CE
- **Tensor shape 계약:** tokens [B,T] → hidden [B,T,D] → MLM [B,T,V], NSP [B,2]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 선택 수와 특수 token 비선택을 assert / 각 corruption 개수를 정확히 검사 / NSP accuracy 출력 및 assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
total_losses, mlm_losses, nsp_losses = [], [], []
for _ in range(110):
    model.train()
    optimizer.zero_grad()
    mlm_logits, nsp_logits, _, _ = model(
        corrupted_ids,
        segment_ids,
    )
    mlm_loss = F.cross_entropy(
        mlm_logits.reshape(-1, VOCAB_SIZE),
        mlm_labels.reshape(-1),
        ignore_index=-100,
    )
    nsp_loss = F.cross_entropy(nsp_logits, nsp_labels)
    total_loss = mlm_loss + nsp_loss
    total_loss.backward()
    optimizer.step()
    total_losses.append(float(total_loss.detach()))
    mlm_losses.append(float(mlm_loss.detach()))
    nsp_losses.append(float(nsp_loss.detach()))

model.eval()
with torch.inference_mode():
    mlm_logits, nsp_logits, _, _ = model(
        corrupted_ids,
        segment_ids,
    )
mlm_predictions = mlm_logits[chosen].argmax(dim=-1)
mlm_accuracy = (mlm_predictions == mlm_labels[chosen]).float().mean().item()
nsp_accuracy = (nsp_logits.argmax(dim=-1) == nsp_labels).float().mean().item()

assert total_losses[-1] < 0.2 * total_losses[0]
assert mlm_accuracy >= 0.8 and nsp_accuracy >= 0.8
print(f"MLM accuracy={mlm_accuracy:.1%}, NSP accuracy={nsp_accuracy:.1%}")

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** Figure 1: GPT와 달리 모든 층에서 좌우 문맥을 함께 조건으로 사용
- **이 셀의 책임:** mask 없는 `BidirectionalSelfAttention`과 오른쪽 token 교란 실험
- **Tensor shape 계약:** tokens [B,T] → hidden [B,T,D] → MLM [B,T,V], NSP [B,2]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** right-context sensitivity가 0보다 큰지 assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
context_a = corrupted_ids[:1].clone()
context_b = context_a.clone()
masked_position = int(selected_positions[0])
assert context_a[0, masked_position] == MASK
context_b[0, masked_position + 1] = 22  # only right context changes
with torch.inference_mode():
    logits_a, _, _, attention_a = model(
        context_a,
        segment_ids[:1],
    )
    logits_b, _, _, _ = model(
        context_b,
        segment_ids[:1],
    )
right_context_sensitivity = (
    (logits_a[0, masked_position] - logits_b[0, masked_position]).abs().max().item()
)
assert right_context_sensitivity > 1e-5
right_context_mass = attention_a[
    0,
    :,
    masked_position,
    masked_position + 1 :,
].sum()
assert right_context_mass > 0
print(f"right-context logit sensitivity={right_context_sensitivity:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.3))
axes[0].plot(mlm_losses, label="MLM")
axes[0].plot(nsp_losses, label="NSP")
axes[0].set(title="BERT pre-training objectives", xlabel="step", ylabel="CE")
axes[0].legend()
axes[1].bar(["MLM", "NSP"], [mlm_accuracy, nsp_accuracy])
axes[1].set(title="training-set metrics", ylabel="accuracy", ylim=(0, 1.05))
plt.tight_layout()
plt.show()

## 해석 체크

1. MLM loss를 모든 위치가 아니라 선택된 위치에만 적용하는 이유는 무엇인가요?
2. 선택 token의 10%를 그대로 두는 경우에도 정답 loss를 계산하는 이유는 무엇인가요?
3. 이 노트북의 training accuracy가 실제 downstream 일반화를 뜻하지 않는 이유를 적으세요.

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `BERT Base/Large는 각각 12/24층, 110M/340M 파라미터로 BooksCorpus와 English Wikipedia(총 33억 단어)에서 100만 step 사전학습됐다. 여기서는 10개 문장 쌍, 1층 24-hidden encoder를 100여 step 암기시키는 구조 검증이다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.